# FluxPipeline - Memory Optimization

Learn how to optimize memory usage and handle GPU limitations.

## What You'll Learn
- Monitor GPU memory usage
- Configure memory settings
- Handle OOM (Out of Memory) errors
- Optimize for different hardware

## Setup and Memory Monitoring

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

import torch
import gc
import matplotlib.pyplot as plt

from pipeline import FluxPipeline
from core import MemoryManager, SeedProfile
from config import setup_environment, logger
from utils import setup_workspace

def print_memory_stats():
    """Print current GPU memory statistics."""
    if not torch.cuda.is_available():
        print("CUDA not available")
        return
    
    allocated = torch.cuda.memory_allocated() / 1024**3
    reserved = torch.cuda.memory_reserved() / 1024**3
    total = torch.cuda.get_device_properties(0).total_memory / 1024**3
    
    print(f"GPU Memory:")
    print(f"  Allocated: {allocated:.2f} GB")
    print(f"  Reserved:  {reserved:.2f} GB")
    print(f"  Total:     {total:.2f} GB")
    print(f"  Free:      {total - reserved:.2f} GB")

print_memory_stats()

## Check Your GPU Capabilities

In [ ]:
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"GPU: {props.name}")
    print(f"Total Memory: {props.total_memory / 1024**3:.2f} GB")
    print(f"CUDA Capability: {props.major}.{props.minor}")
    
    # Recommendations based on memory
    total_gb = props.total_memory / 1024**3
    
    if total_gb >= 16:
        print("\n✅ Your GPU can handle 1024x1024 images comfortably")
        recommended_size = 1024
    elif total_gb >= 8:
        print("\n✅ Your GPU can handle 768x768 images")
        print("⚠️  1024x1024 may work but watch for OOM")
        recommended_size = 768
    else:
        print("\n⚠️  Your GPU has limited memory")
        print("   Recommend 512x512 or smaller")
        recommended_size = 512
    
    print(f"\nRecommended image size: {recommended_size}x{recommended_size}")
else:
    print("CUDA not available - using CPU (will be slow)")
    recommended_size = 512

## Initialize with Memory Management

In [ ]:
setup_environment()
workspace = setup_workspace()

# Initialize pipeline with memory monitoring
pipeline = FluxPipeline(workspace=workspace)

print("\nMemory before model load:")
print_memory_stats()

pipeline.load_model()

print("\nMemory after model load:")
print_memory_stats()

## Memory-Efficient Generation

In [ ]:
def generate_with_monitoring(prompt, **kwargs):
    """Generate image with memory monitoring."""
    print(f"Generating: {prompt[:50]}...")
    
    # Check memory before generation
    if torch.cuda.is_available():
        before = torch.cuda.memory_allocated() / 1024**3
    
    # Generate
    image, seed = pipeline.generate_image(prompt=prompt, **kwargs)
    
    # Check memory after generation
    if torch.cuda.is_available():
        after = torch.cuda.memory_allocated() / 1024**3
        print(f"  Memory used: {after - before:.2f} GB")
    
    return image, seed

# Test with recommended size
test_prompt = "A beautiful sunset over mountains"
image, seed = generate_with_monitoring(
    test_prompt,
    height=recommended_size,
    width=recommended_size,
    num_inference_steps=4
)

if image:
    plt.figure(figsize=(8, 8))
    plt.imshow(image)
    plt.axis('off')
    plt.title(f"{recommended_size}x{recommended_size}")
    plt.show()

## Manual Memory Management

In [ ]:
def clear_memory():
    """Manually clear GPU memory."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
    print("✅ Memory cleared")

# Clear memory
print("Before cleanup:")
print_memory_stats()

clear_memory()

print("\nAfter cleanup:")
print_memory_stats()

## Batch Processing with Memory Management

In [ ]:
def memory_safe_batch(prompts, **gen_kwargs):
    """Generate batch with memory cleanup between images."""
    results = []
    
    for idx, prompt in enumerate(prompts, 1):
        print(f"\n[{idx}/{len(prompts)}] {prompt[:50]}...")
        
        # Generate
        image, seed = pipeline.generate_image(prompt=prompt, **gen_kwargs)
        
        if image:
            # Save immediately to reduce memory
            filename = workspace / f"batch_{idx}_{seed}.png"
            image.save(filename)
            results.append(filename)
            print(f"  ✅ Saved: {filename.name}")
            
            # Clear image from memory
            del image
        
        # Clean up after each image
        clear_memory()
        print_memory_stats()
    
    return results

# Test batch with memory management
batch_prompts = [
    "A serene lake at dawn",
    "A bustling city street",
    "A quiet forest path"
]

saved_files = memory_safe_batch(
    batch_prompts,
    height=512,
    width=512,
    num_inference_steps=4
)

print(f"\n✅ Batch complete! Saved {len(saved_files)} images")

## Handling OOM Errors

If you get Out of Memory errors, try these fixes:

In [ ]:
def generate_with_fallback(prompt, sizes=None, **kwargs):
    """Try generating with progressively smaller sizes if OOM occurs."""
    if sizes is None:
        sizes = [1024, 768, 512, 384]
    
    for size in sizes:
        try:
            print(f"\nTrying {size}x{size}...")
            clear_memory()
            
            image, seed = pipeline.generate_image(
                prompt=prompt,
                height=size,
                width=size,
                **kwargs
            )
            
            print(f"✅ Success at {size}x{size}!")
            return image, seed, size
            
        except RuntimeError as e:
            if "out of memory" in str(e).lower():
                print(f"  ❌ OOM at {size}x{size}, trying smaller...")
                clear_memory()
                continue
            else:
                raise
    
    print("❌ Failed at all sizes")
    return None, None, None

# Test fallback mechanism
test_prompt = "A detailed fantasy castle"
result_image, result_seed, result_size = generate_with_fallback(
    test_prompt,
    num_inference_steps=4
)

if result_image:
    print(f"\nGenerated at {result_size}x{result_size}")
    plt.figure(figsize=(8, 8))
    plt.imshow(result_image)
    plt.axis('off')
    plt.show()

## Memory Optimization Tips

### 1. Reduce Image Size
- Start with 512x512 and increase if memory allows
- Remember: larger images use exponentially more memory

### 2. Clear Memory Regularly
- Call `clear_memory()` between generations
- Delete large variables when done

### 3. Batch Processing
- Save images immediately, don't keep in memory
- Process one at a time if memory limited

### 4. Monitor Usage
- Use `print_memory_stats()` to track usage
- Watch for memory leaks

### 5. System Settings
- Close other GPU applications
- Use appropriate precision (float16 vs float32)

## Memory Usage by Size

In [ ]:
# Test different sizes to see memory usage
if torch.cuda.is_available() and torch.cuda.get_device_properties(0).total_memory / 1024**3 >= 12:
    test_sizes = [384, 512, 768, 1024]
else:
    test_sizes = [384, 512, 768]

memory_usage = []
test_prompt = "A simple landscape"

for size in test_sizes:
    try:
        clear_memory()
        before = torch.cuda.memory_allocated() / 1024**3 if torch.cuda.is_available() else 0
        
        image, _ = pipeline.generate_image(
            prompt=test_prompt,
            height=size,
            width=size,
            num_inference_steps=4
        )
        
        after = torch.cuda.memory_allocated() / 1024**3 if torch.cuda.is_available() else 0
        used = after - before
        
        memory_usage.append((size, used))
        print(f"{size}x{size}: {used:.2f} GB")
        
        del image
        clear_memory()
        
    except RuntimeError as e:
        if "out of memory" in str(e).lower():
            print(f"{size}x{size}: OOM")
            break

# Plot memory usage
if memory_usage:
    sizes, usage = zip(*memory_usage)
    plt.figure(figsize=(10, 6))
    plt.bar([str(s) for s in sizes], usage)
    plt.xlabel('Image Size')
    plt.ylabel('Memory Usage (GB)')
    plt.title('GPU Memory Usage by Image Size')
    plt.grid(axis='y', alpha=0.3)
    plt.show()

## Next Steps

- **[06_seed_management.ipynb](06_seed_management.ipynb)** - Control randomness and variations
- Experiment with sizes that work best for your GPU

In [ ]:
# Final cleanup
del pipeline
clear_memory()
print("\n✅ All done!")
print_memory_stats()